In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
from tqdm import tqdm 
from sklearn.metrics import classification_report, confusion_matrix

from CNN_module.cnn_engine import CNNDetector
from Transformer_module.transformer_engine import TransformerDetector

cnn_checker = CNNDetector()
transformer_checker = TransformerDetector()

test = pd.read_csv("data/test_data.csv")
test_extended = pd.read_csv("data/test_extended.csv")
hard_test = pd.read_csv("data/hard_test.csv")
df = pd.concat([test, test_extended, hard_test], ignore_index=True)

emails = df['text'].tolist()
tags = df['label'].values

n_ham = df[df['label'] == 0].shape[0]
n_spam = df[df['label'] == 1].shape[0]
n = len(df)

cnn_results = []
transformer_results = []
batch_size = 32 

for i in tqdm(range(0, n, batch_size)):
    batch_texts = emails[i : i + batch_size]
    
    batch_cnn_preds = cnn_checker.predict(batch_texts)
    batch_trans_preds = transformer_checker.predict(batch_texts)
    
    cnn_results.extend(batch_cnn_preds)
    transformer_results.extend(batch_trans_preds)

cnn_results = np.array(cnn_results)
transformer_results = np.array(transformer_results)

cnn_binary_preds = (cnn_results > 50).astype(int)
trans_binary_preds = (transformer_results > 50).astype(int)

CNN_counter = np.sum(cnn_binary_preds == tags)
Transformer_counter = np.sum(trans_binary_preds == tags)

CNN_counter_ham = np.sum((cnn_binary_preds == 0) & (tags == 0))
CNN_counter_spam = np.sum((cnn_binary_preds == 1) & (tags == 1))
Transformer_counter_ham = np.sum((trans_binary_preds == 0) & (tags == 0))
Transformer_counter_spam = np.sum((trans_binary_preds == 1) & (tags == 1))

print("\n" + "="*50)
print("I: ACCURACY")
print("="*50)
print(f"CNN đúng tổng: {CNN_counter}/{n} = {CNN_counter/n*100:.2f}%")
print(f"Transformer đúng tổng: {Transformer_counter}/{n} = {Transformer_counter/n*100:.2f}%")

print(f"\n[CNN] Đúng Thư Thường (Ham): {CNN_counter_ham}/{n_ham} = {CNN_counter_ham/n_ham*100:.2f}%")
print(f"[CNN] Đúng Thư Rác (Spam): {CNN_counter_spam}/{n_spam} = {CNN_counter_spam/n_spam*100:.2f}%")

print(f"\n[Transformer] Đúng Thư Thường (Ham): {Transformer_counter_ham}/{n_ham} = {Transformer_counter_ham/n_ham*100:.2f}%")
print(f"[Transformer] Đúng Thư Rác (Spam): {Transformer_counter_spam}/{n_spam} = {Transformer_counter_spam/n_spam*100:.2f}%")


target_names = ['Thư thường (Ham)', 'Thư rác (Spam)']

print("\n" + "="*50)
print("II: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH CNN")
print("="*50)

print("1. CONFUSION MATRIX:")
cnn_cm = confusion_matrix(tags, cnn_binary_preds)
print(f"                     Dự đoán: HAM   Dự đoán: SPAM")
print(f"Thực tế là HAM:        {cnn_cm[0][0]:<10}   {cnn_cm[0][1]:<10}")
print(f"Thực tế là SPAM:       {cnn_cm[1][0]:<10}   {cnn_cm[1][1]:<10}")

print("\n2. CLASSIFICATION REPORT:")
print(classification_report(tags, cnn_binary_preds, target_names=target_names, digits=4))


print("\n" + "="*50)
print("III: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH TRANSFORMER")
print("="*50)

print("1. CONFUSION MATRIX:")
trans_cm = confusion_matrix(tags, trans_binary_preds)
print(f"                     Dự đoán: HAM   Dự đoán: SPAM")
print(f"Thực tế là HAM:        {trans_cm[0][0]:<10}    {trans_cm[0][1]:<10}")
print(f"Thực tế là SPAM:       {trans_cm[1][0]:<10}    {trans_cm[1][1]:<10}")

print("\n2. CLASSIFICATION REPORT:")
print(classification_report(tags, trans_binary_preds, target_names=target_names, digits=4))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: c:\Users\ADMIN\Downloads\Code\.vscode\SPAM_CLASSIFIER\Transformer_module\my_bert_model
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 454/454 [01:35<00:00,  4.77it/s]


I: ACCURACY
CNN đúng tổng: 14356/14508 = 98.95%
Transformer đúng tổng: 14391/14508 = 99.19%

[CNN] Đúng Thư Thường (Ham): 7334/7426 = 98.76%
[CNN] Đúng Thư Rác (Spam): 7022/7082 = 99.15%

[Transformer] Đúng Thư Thường (Ham): 7339/7426 = 98.83%
[Transformer] Đúng Thư Rác (Spam): 7052/7082 = 99.58%

II: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH CNN
1. CONFUSION MATRIX:
                     Dự đoán: HAM   Dự đoán: SPAM
Thực tế là HAM:        7334         92        
Thực tế là SPAM:       60           7022      

2. CLASSIFICATION REPORT:
                  precision    recall  f1-score   support

Thư thường (Ham)     0.9919    0.9876    0.9897      7426
  Thư rác (Spam)     0.9871    0.9915    0.9893      7082

        accuracy                         0.9895     14508
       macro avg     0.9895    0.9896    0.9895     14508
    weighted avg     0.9895    0.9895    0.9895     14508


III: ĐÁNH GIÁ CHUYÊN SÂU CHO MÔ HÌNH TRANSFORMER
1. CONFUSION MATRIX:
                     Dự đoán: HAM   Dự đoán: S